In [1]:
# ============================================================
# STEP 5: TEXT CLEANING + TOKENIZATION (Colab, CPU kaafi hai)
# ============================================================
import os, sys, re
from collections import Counter

# ---------- 1. Repo aur libraries ----------
REPO_URL = "https://github.com/FatimaaShaheen/urdu-asr-subtitles.git"
REPO_DIR = "/content/urdu-asr-subtitles"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

!pip install -q "datasets<4" transformers jiwer

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

import pandas as pd, jiwer
from src.data.clean_text import basic_clean, normalize_for_wer

# ---------- 2. Purani normalization (Step 4 wali) ----------
_D = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")
_P = re.compile(r"[۔،؟!?,.:;\"'()\[\]{}\-–—…٪%]")
def old_norm(t):
    t = t.replace("ي","ی").replace("ى","ی").replace("ك","ک").replace("ه","ہ").replace("ۃ","ہ")
    t = _D.sub("", t); t = _P.sub(" ", t)
    return re.sub(r"\s+"," ",t).strip()

# ---------- 3. Baseline predictions dobara score karo ----------
df = pd.read_csv(f"{REPO_DIR}/docs/results/baseline_small_predictions.csv")
refs, hyps = df["reference"].tolist(), df["prediction"].tolist()

def score(norm):
    r = [norm(x) for x in refs]; h = [norm(x) for x in hyps]
    pairs = [(a, b) for a, b in zip(r, h) if a]
    r, h = zip(*pairs)
    return round(jiwer.wer(list(r), list(h)), 4), round(jiwer.cer(list(r), list(h)), 4)

print("=== whisper-small, EK HI predictions, do normalizations ===")
print("Purani (Step 4):  WER, CER =", score(old_norm))
print("Nayi  (Step 5):   WER, CER =", score(normalize_for_wer))

# ---------- 4. Sab se kharab 5 jumle ----------
rows = []
for r, h in zip(refs, hyps):
    rn, hn = normalize_for_wer(r), normalize_for_wer(h)
    if rn:
        rows.append((jiwer.wer(rn, hn), rn, hn))
rows.sort(reverse=True)
print("\n=== Sab se kharab 5 jumle ===")
for w, rn, hn in rows[:5]:
    print(f"WER {w:.2f}\nREF : {rn}\nPRED: {hn}\n" + "-"*50)

# ---------- 5. Dataset ki vocabulary: safai se pehle aur baad ----------
from src.data.load_dataset import load_fleurs_urdu
train = load_fleurs_urdu("train")

def vocab_of(texts):
    c = Counter()
    for t in texts:
        c.update(t)
    return c

v_raw   = vocab_of(train["raw_transcription"])
v_clean = vocab_of(normalize_for_wer(t) for t in train["transcription"])
print("\nUnique characters (raw):  ", len(v_raw))
print("Unique characters (saaf): ", len(v_clean))
odd = [(c, n) for c, n in v_clean.items()
       if c != " " and not ("\u0600" <= c <= "\u06FF")]
print("Saaf text mein ab bhi gair-Urdu characters:", odd)

# ---------- 6. Whisper ka tokenizer ----------
from transformers import WhisperTokenizer
tok = WhisperTokenizer.from_pretrained("openai/whisper-small")

ur = basic_clean(train["raw_transcription"][0])
en = "The weather is very nice today and we are going to the park"
for name, t in [("Urdu", ur), ("English", en)]:
    ids = tok(t, add_special_tokens=False).input_ids
    n_words = len(t.split())
    print(f"{name}: {n_words} alfaaz -> {len(ids)} tokens "
          f"({len(ids)/n_words:.2f} tokens/lafz)")

sample = train["transcription"][:100]
ur_ratio = sum(len(tok(t, add_special_tokens=False).input_ids) for t in sample) / \
           sum(len(t.split()) for t in sample)
print(f"\n100 Urdu jumlon ka average: {ur_ratio:.2f} tokens per lafz")

# Special tokens: model ko zabaan aur kaam batate hain
tok.set_prefix_tokens(language="urdu", task="transcribe")
ids = tok(ur).input_ids
print("\nShuru ke special tokens:", tok.convert_ids_to_tokens(ids[:4]))
print("Aakhri token:", tok.convert_ids_to_tokens(ids[-1:]))
print("Tokenizer vocabulary size:", tok.vocab_size)

Cloning into '/content/urdu-asr-subtitles'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 72 (delta 25), reused 59 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 2.42 MiB | 10.17 MiB/s, done.
Resolving deltas: 100% (25/25), done.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
=== whisper-small, EK HI predictions, do normalizations ===
Purani (Step 4):  WER, CER = (0.3672, 0.1373)
Nayi  (Step 5):   WER, CER = (0.3631, 0.1362)

=== Sab se kharab 5 jumle ===
WER 0.93
REF : سکوٹرب بس 403 باقاعدگی سے کیبو ڈا روکا میں رکتے ہوئے سنٹارا جاتی ہے
PRED: اسکوٹر برسچ اور ستیس بقائدی سیکیویڈا روکہ میں رکھتے ہوئے سندھارہ کی جاتے ہیں
-------------------------

README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

parquet-data/ur_pk/train-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 1.60GB            

parquet-data/ur_pk/train-00000-of-00001.(…): downloading bytes:           |  0.00B            

parquet-data/ur_pk/validation-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  174MB            

parquet-data/ur_pk/validation-00000-of-0(…): downloading bytes:           |  0.00B            

... (10 outputs left out)

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Urdu: 30 alfaaz -> 99 tokens (3.30 tokens/lafz)
English: 13 alfaaz -> 13 tokens (1.00 tokens/lafz)

100 Urdu jumlon ka average: 2.69 tokens per lafz

Shuru ke special tokens: ['<|startoftranscript|>', '<|ur|>', '<|transcribe|>', '<|notimestamps|>']
Aakhri token: ['<|endoftext|>']
Tokenizer vocabulary size: 50258
